# Panel de resultados — Predicción de demanda de transporte público en Madrid

**TFM · Fase 6 — Dashboard de resultados**

Este cuaderno **no entrena ni recalcula ningún modelo**. Lee los artefactos ya generados en
las fases 3–5b (`data/processed/`, `models/`) y los representa. Toda la lógica vive en
`src/evaluation/`: `dashboard_data.py` (carga y agregación) y `dashboard_figures.py`
(construcción de figuras). Aquí sólo se importa, se conectan los controles y se dibuja.

## Requisito previo

El paquete debe estar instalado en modo editable para que `from src...` funcione desde
`notebooks/`:

```bash
pip install -e .
```

## Exportación de figuras estáticas

Las versiones PNG para la memoria **no** se generan desde este cuaderno (kaleido 1.x
bloquea el kernel al invocarse repetidamente). Se generan con:

```bash
python -m src.evaluation.export_figures
```

Ambos caminos usan los mismos constructores de `dashboard_figures.py`, de modo que las
figuras interactivas y las exportadas no pueden divergir.

In [1]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from src.evaluation.dashboard_data import (
    MODEL_NAMES,
    available_models,
    error_by_day_type,
    load_all_predictions,
    model_comparison_table,
)
from src.evaluation.dashboard_figures import (
    comparison_figure,
    day_type_figure,
    demand_figure,
    importance_group_figure,
    importance_top_figure,
    residual_figure,
)
from src.utils.seed import set_global_seed

# Fija únicamente el orden de presentación para que el render sea reproducible.
# En este cuaderno no se entrena nada.
set_global_seed(42, deterministic_ops=False)

# No se fija pio.renderers.default a "notebook_connected": ese renderer descarga plotly.js
# desde la CDN y bloquea la ejecución headless (nbconvert) sin emitir error. El renderer
# por defecto (mimetype) funciona tanto en Jupyter Lab como en nbconvert.

PREDICTIONS = load_all_predictions()
print(f"{len(PREDICTIONS):,} filas · {PREDICTIONS['model'].nunique()} modelos · "
      f"{PREDICTIONS['date'].min():%Y-%m-%d} → {PREDICTIONS['date'].max():%Y-%m-%d}")
PREDICTIONS.head()

11,435 filas · 11 modelos · 2023-01-29 → 2026-08-02


,date,split,model,y_true,y_pred
0,2023-01-29,train,ensemble_equal,2245335.0,3.106379e+05
1,2023-01-30,train,ensemble_equal,4970743.0,4.277150e+06
2,2023-01-31,train,ensemble_equal,5159406.0,5.024709e+06
3,2023-02-01,train,ensemble_equal,5305779.0,4.983416e+06
4,2023-02-02,train,ensemble_equal,5340342.0,5.257569e+06


## Cómo reutilizar este cuaderno en la memoria

- **Tablas**: cada sección incluye, bajo la vista interactiva, una tabla estática ya
  formateada (cabeceras en español, cifras redondeadas). Se copia directamente desde la
  salida HTML renderizada y Google Docs la pega como **tabla nativa editable**, no como
  imagen — por eso se muestran como `DataFrame` normal y no con estilos `.style`.
- **Pies de figura**: bajo cada figura hay una cita en bloque (`>`) con el texto
  «Figura N. …» listo para copiar literalmente.
- **Imágenes**: los PNG de `reports/figures/` se generan con
  `python -m src.evaluation.export_figures` y usan **el mismo módulo de tema**
  (`src/evaluation/theme.py`) que las figuras interactivas, de modo que una figura copiada
  del cuaderno y la misma figura insertada desde el PNG se ven idénticas.

---
## b. Demanda real vs. predicha

Serie observada y predicha superpuestas. Por defecto sólo **validación y test**: el
conjunto de entrenamiento es *in-sample* para los modelos aprendidos (XGBoost-alone alcanza
allí R² = 0,9929) y superponerlo daría una impresión visual engañosa. La casilla
`Mostrar train` lo revela de forma explícita, y la figura queda rotulada como in-sample.

In [2]:
model_dd = widgets.Dropdown(options=MODEL_NAMES, value="ensemble_equal", description="Modelo:")
split_dd = widgets.Dropdown(options=["val", "test"], value="test", description="Partición:")
train_cb = widgets.Checkbox(value=False, description="Mostrar train (in-sample)")
out_b = widgets.Output()


def _render_b(*_):
    split_dd.options = ["train", "val", "test"] if train_cb.value else ["val", "test"]
    if split_dd.value not in split_dd.options:
        split_dd.value = "test"
    with out_b:
        out_b.clear_output(wait=True)
        if model_dd.value not in available_models(split_dd.value, PREDICTIONS):
            print(f"'{model_dd.value}' no tiene predicciones en '{split_dd.value}' "
                  "(cobertura val/test únicamente).")
            return
        display(demand_figure(model_dd.value, split_dd.value, PREDICTIONS))


for _w in (model_dd, split_dd, train_cb):
    _w.observe(_render_b, names="value")

display(widgets.VBox([widgets.HBox([model_dd, split_dd, train_cb]), out_b]))
_render_b()

> Figura 1. Demanda diaria real y predicha por el modelo ensemble_equal en la partición
> test. Serie observada y predicha superpuestas.

---
## c. Comparación entre modelos

Barras agrupadas de MAE, RMSE, MAPE y R², ordenadas de mejor a peor por MAE. El color
codifica la **familia** del modelo, de modo que el hallazgo central resulte legible de un
vistazo: los dos mejores modelos son *ensembles* de modelos de una sola etapa y **ninguno
utiliza la LSTM**.

In [3]:
split_dd_c = widgets.Dropdown(options=["train", "val", "test"], value="test", description="Partición:")
out_c = widgets.Output()


def _render_c(*_):
    with out_c:
        out_c.clear_output(wait=True)
        display(model_comparison_table(split_dd_c.value, PREDICTIONS).round(4))
        display(comparison_figure(split_dd_c.value, PREDICTIONS))


split_dd_c.observe(_render_c, names="value")
display(widgets.VBox([split_dd_c, out_c]))
_render_c()

In [4]:
# Tabla estática para la memoria — copiar directamente desde la salida HTML
from src.evaluation.report_tables import comparison_table

display(comparison_table("test"))

,Modelo,Familia,n,MAE,RMSE,MAPE (%),R²
0,ensemble_inverse_mae,Ensemble,197,139681,208339,3.11,0.9755
1,ensemble_equal,Ensemble,197,139725,208464,3.13,0.9755
2,xgboost_alone,Una etapa,197,156121,234154,3.30,0.9690
3,sarimax,Una etapa,197,163627,241147,3.85,0.9672
4,hybrid_weighted,Familia LSTM,197,220286,319447,5.17,0.9424
5,hybrid,Familia LSTM,197,234223,328131,5.49,0.9392
6,lstm_alone,Familia LSTM,197,280952,431714,6.62,0.8948
7,seasonal_naive,Baseline ingenuo,197,309817,690480,7.16,0.7308
8,persistence,Baseline ingenuo,197,900570,1390444,21.79,-0.0916
9,moving_average_7,Baseline ingenuo,197,1093679,1260492,28.13,0.1029


> Tabla 1. Comparación de modelos en la partición test, ordenada de mejor a peor por MAE.
>
> Figura 2. Comparación de modelos en la partición test. Barras agrupadas de MAE, RMSE,
> MAPE y R², ordenadas de mejor a peor por MAE. El color codifica la familia del modelo.

---
## d. Error de predicción (residuos)

### Cómo se ve una brecha de sobreajuste

Si la distribución de residuos en **train** es mucho más estrecha que en **test**, el modelo
ha memorizado el periodo de entrenamiento en lugar de generalizar. El caso documentado en
`CLAUDE.md` es **`xgboost_alone`**: R² de 0,9929 en train (MAE ≈ 72.900) frente a un MAE de
156.121 en test — más del doble de error fuera de muestra. Sigue siendo el mejor modelo
individual en test; la brecha indica que la cifra *in-sample* no debe citarse como medida
de rendimiento.

Los modelos derivados de la LSTM (`lstm_alone`, `hybrid`, `hybrid_weighted`) sólo tienen
cobertura en val/test, por lo que para ellos el panel derecho aparece vacío.

In [5]:
model_dd_d = widgets.Dropdown(options=MODEL_NAMES, value="xgboost_alone", description="Modelo:")
out_d = widgets.Output()


def _render_d(*_):
    with out_d:
        out_d.clear_output(wait=True)
        display(residual_figure(model_dd_d.value, "test", PREDICTIONS))


model_dd_d.observe(_render_d, names="value")
display(widgets.VBox([model_dd_d, out_d]))
_render_d()

> Figura 3. Distribución del error de predicción del modelo xgboost_alone en la partición
> test. Si la distribución de residuos en train es mucho más estrecha que en test, el
> modelo ha memorizado el periodo de entrenamiento en lugar de generalizar.

---
## e. Desglose de error por tipo de día

Aquí se comprueba la afirmación central de la arquitectura: que la corrección residual
reduce el error **específicamente en festivos y puentes**, no sólo en agregado. No lo hace:
ambas variantes del híbrido quedan por detrás de SARIMAX, XGBoost-alone y el ensemble en
todos los grupos.

> ⚠️ **Los grupos con n < 10 se dibujan con opacidad reducida y su n anotado bajo el eje.**
> En test, *festivo* tiene n = 5 y *laborable, puente* n = 3: son indicativos, no
> concluyentes. Para afirmaciones sobre dónde se concentra el error, la base sólida es el
> diagnóstico de la Fase 4 (n = 48 y n = 54 sobre el periodo completo).

In [6]:
split_dd_e = widgets.Dropdown(options=["val", "test"], value="test", description="Partición:")
out_e = widgets.Output()


def _render_e(*_):
    with out_e:
        out_e.clear_output(wait=True)
        table = error_by_day_type(split_dd_e.value, PREDICTIONS)
        display(table[table["small_sample"]].round(2))
        display(day_type_figure(split_dd_e.value, predictions=PREDICTIONS))


split_dd_e.observe(_render_e, names="value")
display(widgets.VBox([split_dd_e, out_e]))
_render_e()

In [7]:
from src.evaluation.report_tables import day_type_table

display(day_type_table("test"))

,Tipo de día,n,sarimax,xgboost_alone,hybrid,hybrid_weighted,ensemble_equal
0,Laborable ordinario,133,148297,169488,211581,200326,138628
1,Laborable,136,150493,169845,215729,205438,140352
2,Sábado,27,203910,129381,247793,185918,138805
3,Domingo,29,169129,100200,233778,243096,122591
4,Festivo,5,271410,251553,666572,677429,227037
5,"Laborable, puente",3,247865,185671,399637,432066,216768


> Tabla 2. MAE por tipo de día en la partición test. La columna n indica el tamaño
> muestral de cada grupo.
>
> Figura 4. MAE por tipo de día en la partición test. Los grupos con n < 10 se dibujan con
> opacidad reducida y su n anotado bajo el eje: son indicativos, no concluyentes.

---
## f. Importancia de variables (XGBoost)

Dos paneles: importancia agregada **por grupo de variables** y **top-15 individual**. El
contraste entre modelos es informativo:

- **`xgboost_residual`** (etapa 2 del híbrido) se apoya fuertemente en el calendario —
  `feat_day_type_festivo` es su variable más importante — porque su trabajo es corregir
  exactamente lo que la LSTM univariante no puede observar.
- **`xgboost_alone`** reparte su importancia de otro modo, ya que modela la demanda completa
  y no un residuo.

In [8]:
model_dd_f = widgets.Dropdown(
    options=["xgboost_residual", "xgboost_alone"],
    value="xgboost_residual",
    description="Modelo:",
)
out_f = widgets.Output()


def _render_f(*_):
    with out_f:
        out_f.clear_output(wait=True)
        display(importance_group_figure(model_dd_f.value))
        display(importance_top_figure(model_dd_f.value))


model_dd_f.observe(_render_f, names="value")
display(widgets.VBox([model_dd_f, out_f]))
_render_f()

In [9]:
from src.evaluation.report_tables import (
    feature_importance_table,
    group_importance_table,
    hyperparameters_table,
)

display(hyperparameters_table())
display(group_importance_table("xgboost_residual"))
display(feature_importance_table("xgboost_residual"))

,Modelo,Filas entrenamiento,max_depth,n_estimators,learning_rate,min_child_weight
0,xgboost_residual,552,5,300,0.01,3
1,xgboost_alone,889,5,100,0.05,10


,Grupo,N.º variables,Ganancia,Peso (%)
0,weather,105,0.5157,51.57
1,calendar,11,0.1726,17.26
2,lag_operator,28,0.1723,17.23
3,lag_total,7,0.0592,5.92
4,rolling,4,0.0405,4.05
5,fourier_annual,4,0.0320,3.20
6,fourier_weekly,2,0.0077,0.77


,#,Variable,Grupo,Ganancia
0,1,feat_day_type_festivo,calendar,0.1254
1,2,feat_apparent_temperature_max_lag_7,weather,0.0759
2,3,feat_temperature_2m_mean_lag_3,weather,0.0337
3,4,feat_carretera_lag_3,lag_operator,0.0319
4,5,feat_day_type_laborable,calendar,0.0305
5,6,feat_total_lag_7,lag_total,0.0249
6,7,feat_total_roll_std_28,rolling,0.0217
7,8,feat_apparent_temperature_min_lag_3,weather,0.0200
8,9,feat_apparent_temperature_max_lag_3,weather,0.0191
9,10,feat_temperature_2m_min_roll_mean_7,weather,0.0189


> Tabla 3. Hiperparámetros seleccionados de ambos modelos XGBoost (fases 5 y 5b).
>
> Tabla 4. Importancia por grupo de variables del modelo xgboost_residual.
>
> Tabla 5. Top-15 variables individuales por ganancia del modelo xgboost_residual.
>
> Figura 5. Importancia agregada por grupo de variables del modelo xgboost_residual,
> medida como ganancia.
>
> Figura 6. Top-15 variables individuales por ganancia del modelo xgboost_residual.

---
## g. Resumen ejecutivo

La arquitectura híbrida residual propuesta (LSTM en la etapa 1, XGBoost sobre los residuos
en la etapa 2) **no supera a los modelos de una sola etapa** con los que se compara. En la
partición de test, el híbrido obtiene un MAE de 234.223 frente a 163.627 de SARIMAX y
156.121 de XGBoost aplicado directamente sobre la demanda; en validación la ordenación es la
misma. El resultado se mantiene en la variante con ponderación de días irregulares
(Fase 5b, MAE 220.286), que además no mejoró los festivos y puentes que pretendía corregir.

El mejor modelo del proyecto es un **ensemble sencillo al 50/50 entre SARIMAX y
XGBoost-alone**, con un MAE de 139.725 en test — un 10,5 % por debajo del mejor de sus dos
componentes. No se trata del promedio aritmético de sus errores, sino de reducción de
varianza: ambos modelos fallan en días distintos (SARIMAX es mejor en laborables ordinarios,
XGBoost-alone en fines de semana y festivos), de modo que promediarlos recupera las dos
fortalezas. **Este modelo no utiliza la LSTM en absoluto.**

La etapa 2 sí funciona como mecanismo: reduce el error de la etapa 1 un 31,7 % en validación
y, en días festivos, de 1.585.388 a 666.572 (−58 %). El problema está en la etapa 1: una
LSTM univariante sobre ~860 observaciones diarias no puede observar el calendario y queda
por debajo incluso del ingenuo estacional. La corrección residual rescata un modelo base
débil, pero no lo lleva por encima de modelos de una sola etapa bien construidos.

Dos salvedades sobre la lectura de las cifras: en test, *festivo* tiene n = 5 y *puente*
n = 3, por lo que esas celdas son indicativas y no concluyentes; y el R² de 0,9929 de
`xgboost_alone` en entrenamiento es *in-sample* y no debe citarse como rendimiento.

**Conclusión defendible:** para demanda agregada diaria con este tamaño muestral, un modelo
de una sola etapa sobre variables exógenas bien construidas —o el promedio de dos de ellos—
supera a una arquitectura híbrida residual secuencial. El valor de la arquitectura híbrida
se limita a rescatar una etapa temporal débil.